In [1]:
from tools.scan_repo import scan_repo
from tools.read_file import read_file
from tools.write_file import write_file
from tools.notebook_sandbox import NotebookSandbox
from tools.record_attempt import record_attempt
from tools.execute_file import execute_file
from utils.display_functions import pretty_print_chat_completion, pretty_print_chat_completion_html

from dotenv import load_dotenv
import os
import json
import re

import litellm

sandbox = NotebookSandbox()

load_dotenv()
litellm.success_callback = ["langsmith"]
litellm.failure_callback = ["langsmith"]
client = litellm.LiteLLM()

[SANDBOX] Initializing persistent Python kernel...
[SANDBOX] Isolated playground environment ready.


In [2]:
#for now don't focus on writing the code back into the file, instead make it output the report along with the buggy code.

In [3]:
class Agent:
    def __init__(self, system=""):
        self.system = system
        self.messages = []
        if self.system:
            self.messages.append({"role": "system", "content": system})

    def __call__(self, message):
        self.messages.append({"role": "user", "content": message})
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result

    def execute(self):
        completion = client.chat.completions.create(
                        model="gemini/gemini-2.5-flash", 
                        temperature=0,
                        messages=self.messages)
        return completion.choices[0].message.content

In [4]:
system_prompt = """
You are a system architect, you will be given output of the buggy code
you will run in a loop of Thought, action, PAUSE, Observation, at the end you output ANSWER code that fixes the bug along with what you tried to solve the bug
if the attempt solved the bug, ouput what caused the bug, what solved the bug, and how your solution solved the bug,
if the attempt didn't solve the bug, output what caused the bug and what solution you tried and why it failed along with the failing output.

available tools:

scan_repo: 
e.g. scan_repo: None
scans the default target repository and returns list that consists of all the file names the repository has.

read_file:
e.g. read_file: file_name.ext
reads all the contents of the file including jupyter notebook and returns string that consists of file content.

write_file: 
e.g. write_file: file_name.ext, content
writes content inside a file, this tool can be used to write the content inside a file.

record_attempt:
e.g. record_attempt: hypothesis, code_changed, result, why_it_failed
this is used to record all the attempts you done to solve the bug,
why_it_failed(argument): use this only when attempt has failed else pass None.

notebook_sandbox:
e.g. notebook_sanbox: 
it is a class that creates a jupyter notebook sandbox to experiment with the buggy code, it also has various tools:
    add_cell: code
    Appends a new Python code cell to the bottom of the virtual Jupyter notebook. Use this to write new experimental code or define functions. Returns the new cell's ID.
    execute_cell: cell_id
    Executes a specific code cell in the virtual notebook by its ID. Variables and functions defined here will persist in memory for future cells. Always check the output for runtime errors.
    edit_cell: cell_id, new_code
    Overwrites the Python code in an existing cell. Use this specifically to fix syntax or runtime errors in a cell that previously failed.
    get_sandbox_state: None
    Reads the entire virtual notebook. Returns a formatted string showing all current code cells, their IDs, and their last execution outputs. Use this if you lose track of the notebook's state or need to review past outputs.

execute_file:
e.g. execute_file: file_name.ext, timeout = 60
this is used to execute the file, returns string that contains output of the file executed,
timeout(argument): by default this is set to 60 secs but if you think the file needs more time to execute based on the contents of the file then you can modify this accordingly.

when you are about to create a python code to fix the bug you should write your code *strictly* in this format:

    <execute_python>
    # valid python code here
    </execute_python>
    
example session:

user_question: Application is crashing with a "TypeError: 'NoneType' object is not subscriptable" error.
Thought: I need to find the files in this repository.
Action: scan_repo:
PAUSE
you will be called again with:
Observation: ['data_processor.py']

Thought: user reported "TypeError: 'NoneType' object is not subscriptable" error. Given the repository structure, 'data_processor.py' is likely the file to perform a calculation and produce the error.
Action: read_file: data_processor.py
PAUSE

you will be called again with:
Observation: def get_average_score(scores):
    if not scores:
        return 0
    total = sum(scores)
    return total / len(scores)

def get_highest_score(scores):
    sorted_scores = scores.sort()
    return sorted_scores[-1]

if __name__ == "__main__":
    math_scores = [85, 92, 78, 99, 88]
    print(f"Average: {get_average_score(math_scores)}")
    print(f"Highest: {get_highest_score(math_scores)}")

Thought: I see the bug. I will create a notebook sandbox first.
Action: notebook_sandbox: 
PAUSE

you will be called again with:
Observation: notebook sandbox successfully created.

Thought: I will try method1: change scores.sort() to sort the list in-place and then return the last element. Let me test this logic in the sandbox first to ensure it outputs the correct highest score (99) without crashing.
Action: add_cell: <execute_python>[fix_code]</execute_python>
PAUSE
make sure to write the entire fix code instead of writting 'fix_code' or just function name
you will be called again with:
Observation: cell created with cell_id = [cell_id], content = [fix_code]

Thought: Now I will execute the cell.
Action: execute_cell: cell_id
PAUSE

you will be called again with:
Observation: output: [stdout]

Thought: Output confirms that the fix is working, I will reflect the changes back into the main file,
Action: write_file: [code]
PAUSE

you will be called again with:
Observation: file successfully changed.

then output:
Answer: bug is caused by [bug_code_part], fix that worked [fix_code], how it worked [hypothesis].
""".strip()

In [5]:
available_tools = {
    "read_file": read_file,
    "scan_repo": scan_repo,
    "write_file": write_file,
    "execute_file": execute_file,
    "notebook_sandbox": NotebookSandbox,
    "record_attempt": record_attempt,
    "add_cell": sandbox.add_cell,
    "edit_cell": sandbox.edit_cell,
    "execute_cell": sandbox.execute_cell,
    "get_sandbox_state": sandbox.get_sandbox_state
}
#I want the LLM to record the attempts it tried using record_attempt so that it doesn't repeat the same method again and also LLM knows the attempts it tried in the previous loop.

In [6]:
action_re = re.compile(r"Action:\s*(\w+):\s*(.*)PAUSE", re.DOTALL)


In [7]:
response = """LLM: Thought: I have created the sandbox. Now I will add the corrected `calculate_discount` function to a cell and test it with the provided values (price=100, percent=10).
Action: add_cell: <execute_python>
def calculate_discount(price, percent):
    discount = price * percent / 100 # Corrected division
    return discount # Return the discount amount
print(calculate_discount(100, 10))
</execute_python>
PAUSE"""
tool = action_re.search(response)
func, func_arg = tool.groups()
print(func, func_arg)
if func_arg.strip() != "None":
    print(f"calling: {func} on {func_arg}")
else:
    print(f"calling: {func}")
    
        
# if tool is not None:
#     func, func_arg = action_re.search(response).groups()
#             #4. call the tool
#     if func in available_tools:
#         if func_arg is not None: 
#             print(f"calling: {func} on input {func_arg}")
#             result = available_tools[func](func_arg)
#                     #5. send the result back to the LLM
#             next_prompt = f"Observation: {result}"
#             print(next_prompt)
#         else:
#             print(f"calling: {func}")
#             result = available_tools[func]()
#             next_prompt = f"Observation: {result}"
#             print(next_prompt)
#     else:
#         next_prompt = f"tool {func} is not available"
#         print(next_prompt)

add_cell <execute_python>
def calculate_discount(price, percent):
    discount = price * percent / 100 # Corrected division
    return discount # Return the discount amount
print(calculate_discount(100, 10))
</execute_python>

calling: add_cell on <execute_python>
def calculate_discount(price, percent):
    discount = price * percent / 100 # Corrected division
    return discount # Return the discount amount
print(calculate_discount(100, 10))
</execute_python>



In [8]:
def query(user_prompt: str, max_steps = 10):
    #1. initialize
    bot = Agent(system_prompt)
    next_prompt = user_prompt
    for i in range(1, max_steps+1):
        print(f"-------- step {i} --------")
        #2. call LLM on user prompt or tool result
        response = bot(next_prompt)
        print(f"LLM: {response}")
        #3. check for tools
        tool = action_re.search(response)
        
        if tool is not None:
            func, func_arg = action_re.search(response).groups()
            func, func_arg = func.strip(), func_arg.strip() #temporary fix
            #4. call the tool
            if func in available_tools:
                if func_arg != "None": 
                    print(f"calling: {func} on input {func_arg}")
                    result = available_tools[func](func_arg)
                    #5. send the result back to the LLM
                    next_prompt = f"Observation: {result}"
                    print(next_prompt)
                else:
                    print(f"calling: {func}")
                    result = available_tools[func]()
                    next_prompt = f"Observation: {result}"
                    print(next_prompt)
            else:
                next_prompt = f"tool {func} is not available"
                print(next_prompt)
        else:
            #end the loop if LLM wants no tool to access which means it reached it conclusion
            return "END OF LOOP"

In [10]:
user_prompt = "I am getting 0 as discount value for 100 rupees when 10 percent of 100 is 10"
query(user_prompt)

-------- step 1 --------

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



RateLimitError: litellm.RateLimitError: litellm.RateLimitError: geminiException - {
  "error": {
    "code": 429,
    "message": "You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 49.787412555s.",
    "status": "RESOURCE_EXHAUSTED",
    "details": [
      {
        "@type": "type.googleapis.com/google.rpc.Help",
        "links": [
          {
            "description": "Learn more about Gemini API quotas",
            "url": "https://ai.google.dev/gemini-api/docs/rate-limits"
          }
        ]
      },
      {
        "@type": "type.googleapis.com/google.rpc.QuotaFailure",
        "violations": [
          {
            "quotaMetric": "generati

In [ ]:
"""
For me:
1. I am exhausting the free LLM API keys by just fixing the errors -> sol: try to separate and execute each code block individually:
    a. be absolutely sure that tools are working properly by passing in the inputs like the LLM would.
    b. while automating the workflow, test each individual blocks manually and be absolutely sure that they are working.
    c. **if you hit an error in between the workflow then just try to run that part instead of the entire workflow.
2. Always start with simple loop, don't think much about the edge cases for now, implement the simple workflow then think about the edge cases and write the try/exception clause and if/else clause.
"""